In [1]:
import subprocess
import time
import os
import json
import datetime
import tempfile
import random
import shutil
import asyncio
import signal
from urllib.parse import urlparse, parse_qs


# ==============================================================================
# 0. ONE-TIME INSTALL  (parallel + timed, to keep Kaggle start-up short)
# ==============================================================================

import threading
import importlib.util

_T0 = time.time()


def _log(msg):
    print(f"[setup {time.time() - _T0:4.0f}s] {msg}")


def _sh(cmd):
    """Run a shell command quietly; returns the CompletedProcess."""
    return subprocess.run(
        cmd, shell=True, executable="/bin/bash",
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True
    )


def is_installed(cmd):
    return shutil.which(cmd) is not None


def _has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


# command -> apt package that provides it
_CMD_TO_PKG = {
    "Xvfb": "xvfb",
    "ffmpeg": "ffmpeg",
    "pulseaudio": "pulseaudio",
    "pactl": "pulseaudio-utils",
    "matchbox-window-manager": "matchbox-window-manager",
    "wmctrl": "wmctrl",
}
_missing_pkgs = sorted({pkg for cmd, pkg in _CMD_TO_PKG.items() if not is_installed(cmd)})
if _missing_pkgs:
    _missing_pkgs += ["dbus-x11", "xdotool"]

_state = {"browser_installed_now": False}


def _install_system():
    if not _missing_pkgs:
        _log("System packages already present - skipping apt.")
        return
    _log(f"apt: installing {', '.join(_missing_pkgs)} ...")
    pk = " ".join(_missing_pkgs)
    base = "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
    r = _sh(f"apt-get update -y -qq && {base}--no-install-recommends {pk}")
    if r.returncode != 0:
        _log("Lean apt install failed - retrying with full dependencies...")
        r = _sh(f"apt-get update -y -qq && {base}{pk}")
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("apt-get install failed.")
    _log("apt: done.")


def _install_python():
    pip_pkgs = []
    if not _has_module("playwright"):
        pip_pkgs.append("playwright")
    if not _has_module("requests"):
        pip_pkgs.append("requests")
    if not _has_module("httpx"):
        pip_pkgs.append("httpx")
    if not (_has_module("googleapiclient") and _has_module("google.oauth2")):
        pip_pkgs += ["google-api-python-client", "google-auth", "google-auth-oauthlib"]
    if not _has_module("nepali_datetime"):
        pip_pkgs.append("nepali-datetime")

    if pip_pkgs:
        _log(f"pip: installing {', '.join(pip_pkgs)} ...")
        r = _sh("pip install -q " + " ".join(pip_pkgs))
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("pip install failed.")
        _log("pip: done.")
    else:
        _log("Python packages already present - skipping pip.")

    chk = subprocess.run(
        "playwright install --dry-run chromium 2>&1 | "
        "grep -qi 'is already downloaded' && echo YES || echo NO",
        shell=True, capture_output=True, text=True, executable="/bin/bash"
    )
    if "YES" not in chk.stdout:
        _log("playwright: downloading Chromium ...")
        r = _sh("playwright install chromium")
        if r.returncode != 0:
            print((r.stderr or "")[-1500:])
            raise RuntimeError("playwright install chromium failed.")
        _state["browser_installed_now"] = True
        _log("playwright: Chromium downloaded.")
    else:
        _log("Chromium already present - skipping download.")


_errors = []


def _guard(fn):
    def runner():
        try:
            fn()
        except Exception as e:
            _errors.append(f"{fn.__name__}: {e}")
    return runner


# apt and pip/Chromium downloads use different resources, so run them together
_threads = [
    threading.Thread(target=_guard(_install_system)),
    threading.Thread(target=_guard(_install_python)),
]
for _t in _threads:
    _t.start()
for _t in _threads:
    _t.join()

if _errors:
    raise RuntimeError("Setup failed: " + " | ".join(_errors))

# Chromium's system libraries (needs apt, so it runs after the parallel part)
if _state["browser_installed_now"]:
    _log("playwright: installing Chromium system libraries ...")
    _r = _sh("playwright install-deps chromium")
    if _r.returncode != 0:
        print((_r.stderr or "")[-1500:])
    _log("playwright: system libraries done.")

_log("Setup finished.")

import httpx
import nepali_datetime
import requests
from playwright.async_api import async_playwright


# ==============================================================================
# 1. PROCESS VARIABLES
# ==============================================================================

ffmpeg_proc = None
wm_proc = None
xvfb_proc = None
pulseaudio_proc = None

live_video_id = None      # Facebook live id (set when live is created)
flag_was_set = False      # True once we have written the "playing" flag = True

yt_service = None         # YouTube API client (set when YouTube is used)
yt_broadcast_id = None    # YouTube broadcast id (set when broadcast is created)


# ==============================================================================
# 2. CLEANUP
# ==============================================================================

print("[0/5] Cleaning previous display/stream processes...")

for name in ["Xvfb", "chromium", "chrome", "ffmpeg", "matchbox-window-manager"]:
    subprocess.run(f"pkill -9 -x {name}", shell=True)

time.sleep(2)


# ==============================================================================
# 3. CONFIGURATION
# ==============================================================================

# "both"   = stream live AND save a recording
# "live"   = stream only
# "record" = save a recording only (good for testing, no Facebook/YouTube needed)
MODE = "both"

# Which platform(s) to stream to when MODE is "live" or "both":
#   "both"     = Facebook AND YouTube   (default)
#   "facebook" = Facebook only
#   "youtube"  = YouTube only
STREAM_TO = "both"

# Recording is saved inside Kaggle's working folder (/kaggle/working)
RECORD_PATH = f"/kaggle/working/recording_{int(time.time())}.mkv"
os.makedirs(os.path.dirname(RECORD_PATH), exist_ok=True)

# Upload recordings to your server when finished (record / both modes)
UPLOAD_ENABLED = True
UPLOAD_URL = "https://mukesh-4e2a.onrender.com/upload-video"

# --- FACEBOOK -----------------------------------------------------------------
PAGE_ID = "103517251052640"

# --- YOUTUBE ------------------------------------------------------------------
# "public", "unlisted" or "private"
YOUTUBE_PRIVACY = "public"
YOUTUBE_SCOPES = ["https://www.googleapis.com/auth/youtube"]
# ------------------------------------------------------------------------------

STREAM_TO = STREAM_TO.lower().strip()
if STREAM_TO not in ("both", "facebook", "youtube"):
    raise ValueError('STREAM_TO must be "both", "facebook" or "youtube".')

USE_FACEBOOK = MODE in ("live", "both") and STREAM_TO in ("both", "facebook")
USE_YOUTUBE = MODE in ("live", "both") and STREAM_TO in ("both", "youtube")


def load_secret(name):
    """Read a secret from Kaggle Secrets, then an attached dataset, then env vars."""
    # 1. Kaggle Secrets (works when you run manually in the editor)
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass

    # 2. secrets.json inside an attached private dataset (works for API-triggered runs)
    for path in glob.glob("/kaggle/input/**/secrets.json", recursive=True):
        try:
            with open(path, encoding="utf-8") as f:
                data = json.load(f)
            if name in data:
                value = data[name]
                return value if isinstance(value, str) else json.dumps(value)
        except Exception:
            continue

    # 3. Environment variable
    return os.environ.get(name)


# Load Facebook token from Kaggle Secrets (key name: FB_PAGE_ACCESS_TOKEN)
PAGE_ACCESS_TOKEN = None
if USE_FACEBOOK:
    PAGE_ACCESS_TOKEN = load_secret("FB_PAGE_ACCESS_TOKEN")

    if not PAGE_ACCESS_TOKEN:
        raise RuntimeError(
            "FB_PAGE_ACCESS_TOKEN not found. Add it in Kaggle Secrets "
            "(Add-ons > Secrets) and attach it to this notebook."
        )

WEBSITE_URL = "https://mukeshpokharel-gorkhapatra.static.hf.space/"
# WEBSITE_URL = "https://mukeshpokharel-graphicalquiz.static.hf.space/"

# ------------------------------------------------------------------------------
# TITLES / DESCRIPTIONS (built automatically from the date in WEBSITE_URL)
# ------------------------------------------------------------------------------

# If the URL has no ?date=..., use today's date (Nepal time).
# Set to False if you want a title without any date in that case.
USE_TODAY_IF_NO_DATE = True

NEPALI_MONTHS = ["बैशाख", "जेठ", "असार", "श्रावण", "भदौ", "असोज",
                 "कार्तिक", "मंसिर", "पौष", "माघ", "फाल्गुन", "चैत्र"]


def to_nepali_digits(n):
    return str(n).translate(str.maketrans("0123456789", "०१२३४५६७८९"))


def get_date_label(url):
    """Returns e.g. 'असोज १४' (Nepali date), or None if no date should be used."""
    raw = (parse_qs(urlparse(url).query).get("date") or [None])[0]
    d = None
    if raw:
        try:
            d = datetime.date.fromisoformat(raw.strip())
        except ValueError:
            print(f"[title] Could not read date '{raw}' from URL.")
    if d is None and USE_TODAY_IF_NO_DATE:
        # Kaggle runs in UTC; Nepal is UTC+5:45
        d = (datetime.datetime.now(datetime.timezone.utc)
             + datetime.timedelta(hours=5, minutes=45)).date()
    if d is None:
        return None
    try:
        bs = nepali_datetime.date.from_datetime_date(d)
        return f"{NEPALI_MONTHS[bs.month - 1]} {to_nepali_digits(bs.day)}"
    except Exception as e:
        print(f"[title] Nepali date conversion failed ({e}); using English date.")
        return d.strftime("%b %d")


DATE_LABEL = get_date_label(WEBSITE_URL)

if DATE_LABEL:
    TOPIC = f"गोरखापत्र {DATE_LABEL} मा प्रकाशित सामान्य ज्ञान GK"
    TOPIC_INLINE = f"गोरखापत्र {DATE_LABEL} मा प्रकाशित"
else:
    TOPIC = "गोरखापत्रमा प्रकाशित सामान्य ज्ञान GK"
    TOPIC_INLINE = "गोरखापत्रमा प्रकाशित"

# To include "समसामयिक" in the title, use these instead:
#   TOPIC = f"गोरखापत्र {DATE_LABEL} मा प्रकाशित समसामयिक र सामान्य ज्ञान GK"
#   TOPIC = "गोरखापत्रमा प्रकाशित समसामयिक र सामान्य ज्ञान GK"

# ---- FACEBOOK ----
TITLE = TOPIC
DESCRIPTION = (
    f"📰 {TOPIC_INLINE} समाचारमा आधारित समसामयिक घटना (Current Affairs) तथा सामान्य ज्ञान (GK) प्रश्नोत्तर।\n\n"
    "✅ समसामयिक घटना\n"
    "✅ पुरस्कार तथा सम्मान\n"
    "✅ इतिहास\n"
    "✅ सामान्य ज्ञान\n\n"
    "लोकसेवा, शिक्षक सेवा, प्रहरी, सेना तथा अन्य सरकारी परीक्षाको तयारी गर्नेहरूका लागि उपयोगी।\n\n"
    "👍 Like  💬 Comment  🔔 Follow गर्न नबिर्सनुहोस्।\n\n"
    "#गोरखापत्र #समसामयिक #सामान्यज्ञान #CurrentAffairs #GK #लोकसेवा #NepalGK"
)

# ---- YOUTUBE ----
YT_TITLE = TOPIC[:100]
YT_DESCRIPTION = (
    f"📰 {TOPIC_INLINE} समाचारमा आधारित समसामयिक घटना (Current Affairs) तथा सामान्य ज्ञान (GK) प्रश्नोत्तर, लाइभमा।\n\n"
    "यस लाइभमा समेटिने विषय:\n"
    "• समसामयिक घटना (Current Affairs)\n"
    "• पुरस्कार तथा सम्मान\n"
    "• इतिहास\n"
    "• सामान्य ज्ञान\n\n"
    "लोकसेवा, शिक्षक सेवा, प्रहरी, सेना तथा अन्य सरकारी परीक्षाको तयारी गर्नेहरूका लागि उपयोगी।\n\n"
    "🔔 च्यानल Subscribe गर्नुहोस् र लाइभमा आफ्नो उत्तर Comment गर्नुहोस्।\n\n"
    "#गोरखापत्र #समसामयिक #सामान्यज्ञान #CurrentAffairs #GK #लोकसेवा #NepalGK"
)
YT_TAGS = [
    "गोरखापत्र", "समसामयिक घटना", "सामान्य ज्ञान", "GK", "Nepal GK",
    "current affairs Nepal", "Nepali current affairs", "daily current affairs Nepal",
    "Gorkhapatra", "लोकसेवा", "लोकसेवा तयारी", "loksewa", "loksewa preparation",
    "general knowledge Nepal", "GK quiz Nepal", "Nepali GK",
    "पुरस्कार", "इतिहास", "Nepal history GK", "awards and prizes GK",
]
YT_CATEGORY_ID = "27"   # 27 = Education

print(f"Title: {TOPIC}")

# ------------------------------------------------------------------------------

WIDTH = 1080
HEIGHT = 1350
FRAMERATE = 30

# --- STREAM START TIMING ------------------------------------------------------
# All the slow setup (Facebook / YouTube API calls) now happens BEFORE the
# website is opened. Then the website starts loading and FFmpeg starts exactly
# this many seconds after the page starts loading, so the stream begins right
# where your page's content begins (no extra screen is streamed before it).
# Counted from the moment the page starts loading. FFmpeg itself needs about
# 1 second to start, so lower this by 1 if you want it even tighter.
STREAM_START_DELAY_SECONDS = 10
# ------------------------------------------------------------------------------

DISPLAY_NUM = f":{random.randint(50, 98)}"

# --- FIRESTORE "PLAYING" FLAG (set True by this script, set False by website) --
FIRESTORE_URL = (
    "https://firestore.googleapis.com/v1/projects/loksewa-info/"
    "databases/(default)/documents/app_settings/controls"
)
FIRESTORE_FIELD = "gorkhapatrarecordsplaying"

CONFIRM_DELAY_SECONDS = 7        # wait after setting True before reading back
CONFIRM_RETRY_ATTEMPTS = 3       # how many times to try set + confirm
POLL_INTERVAL_SECONDS = 60       # check the flag every minute
END_GRACE_SECONDS = 3            # small pause after flag goes False before stopping

# Safety net: stop no matter what after this long (in case the website crashes
# before it ever sets the flag to False)
HARD_SAFETY_CAP_MINUTES = 120
# ------------------------------------------------------------------------------


# ==============================================================================
# 3b. FIRESTORE HELPERS
# ==============================================================================

def set_live_status(value: bool) -> bool:
    try:
        r = requests.patch(
            f"{FIRESTORE_URL}?updateMask.fieldPaths={FIRESTORE_FIELD}",
            json={"fields": {FIRESTORE_FIELD: {"booleanValue": value}}},
            timeout=10,
        )
        r.raise_for_status()
        return True
    except Exception as e:
        print(f"[flag] Failed to write value={value}: {e}")
        return False


def get_live_status():
    """Returns True / False, or None if it couldn't be read."""
    try:
        r = requests.get(FIRESTORE_URL, timeout=10)
        r.raise_for_status()
        data = r.json()
        return data.get("fields", {}).get(FIRESTORE_FIELD, {}).get("booleanValue", None)
    except Exception as e:
        print(f"[flag] Failed to read value: {e}")
        return None


def set_and_confirm_flag_true() -> bool:
    """Set the flag True, then read it back to confirm. Retries a few times."""
    global flag_was_set
    for attempt in range(1, CONFIRM_RETRY_ATTEMPTS + 1):
        print(f"[flag] Setting {FIRESTORE_FIELD}=True (attempt {attempt}/{CONFIRM_RETRY_ATTEMPTS})...")
        if set_live_status(True):
            flag_was_set = True
        time.sleep(CONFIRM_DELAY_SECONDS)
        if get_live_status() is True:
            print("[flag] ✅ Confirmed: flag is True.")
            return True
        print("[flag] Could not confirm flag is True yet.")
    return False


# ==============================================================================
# 3c. UPLOAD HELPER
# ==============================================================================

def truncate_text(text, max_len, suffix=""):
    text = (text or "").strip()
    if len(text) <= max_len:
        return text
    return text[:max_len - len(suffix)] + suffix


def upload_video(file_path, description, title=None):
    if not os.path.isfile(file_path) or os.path.getsize(file_path) == 0:
        print(f"❌ Upload skipped: recording is missing or empty: {file_path}")
        return False

    if not title:
        title = description
    # NOTE: your server may cut titles; change 30 to 100 if it allows longer ones
    title = truncate_text(title, 30, "")

    data = {"description": description, "title": title}

    print(f"\n⬆️ Uploading to {UPLOAD_URL} ...")
    try:
        # httpx streams the file from disk, so large videos don't fill RAM
        with httpx.Client(timeout=httpx.Timeout(900.0, connect=30.0)) as client:
            with open(file_path, "rb") as f:
                files = {"video": ("video.mp4", f, "video/mp4")}
                res = client.post(UPLOAD_URL, files=files, data=data)
        if 200 <= res.status_code < 300:
            print("✅ Uploaded successfully!")
            return True
        print(f"❌ Upload failed: {res.status_code} | {res.text[:2000]}")
    except Exception as e:
        print(f"⚠️ Upload error: {e}")
    return False


# ==============================================================================
# 3d. FACEBOOK HELPER (logic unchanged, just wrapped in a function)
# ==============================================================================

def create_facebook_live():
    """Creates the Facebook live video. Returns the secure stream URL."""
    global live_video_id

    print("\n[3/5] Contacting Facebook Graph API...")

    fb_api_url = f"https://graph.facebook.com/v20.0/{PAGE_ID}/live_videos"

    payload = {
        "title": TITLE,
        "description": DESCRIPTION,
        "status": "LIVE_NOW",
        "access_token": PAGE_ACCESS_TOKEN
    }

    response = requests.post(fb_api_url, data=payload, timeout=60)

    print("Facebook HTTP status:", response.status_code)

    res_data = response.json()

    if "secure_stream_url" not in res_data:
        print("\n❌ FAILED TO CREATE FACEBOOK LIVE STREAM")
        print("Facebook Response:")
        print(res_data)
        raise RuntimeError("Facebook Live broadcast creation failed.")

    fb_stream_url = res_data["secure_stream_url"]
    live_video_id = res_data["id"]

    print("\n✅ Facebook Live Stream Created!")
    print(f"Facebook Live ID: {live_video_id}")

    return fb_stream_url


# ==============================================================================
# 3e. YOUTUBE HELPERS
# ==============================================================================

def build_youtube_service():
    """Builds an authorized YouTube API client from the Kaggle secrets."""
    from google.oauth2.credentials import Credentials
    from google.auth.transport.requests import Request
    from googleapiclient.discovery import build

    token_raw = load_secret("TOKEN_JSON")
    secrets_raw = load_secret("CLIENT_SECRETS_JSON")

    if not token_raw:
        raise RuntimeError("TOKEN_JSON not found in Kaggle Secrets (Add-ons > Secrets).")

    token_info = json.loads(token_raw)

    # Fill in client id/secret from CLIENT_SECRETS_JSON if the token lacks them
    if secrets_raw:
        cs = json.loads(secrets_raw)
        cs = cs.get("installed") or cs.get("web") or cs
        if not token_info.get("client_id"):
            token_info["client_id"] = cs.get("client_id")
        if not token_info.get("client_secret"):
            token_info["client_secret"] = cs.get("client_secret")
        if not token_info.get("token_uri"):
            token_info["token_uri"] = cs.get("token_uri", "https://oauth2.googleapis.com/token")

    scopes = token_info.get("scopes") or YOUTUBE_SCOPES
    creds = Credentials.from_authorized_user_info(token_info, scopes)

    if not creds.valid:
        creds.refresh(Request())

    return build("youtube", "v3", credentials=creds, cache_discovery=False)


def update_youtube_metadata():
    """
    Sets tags, category and language on the live video. (The broadcast API only
    accepts title/description, so tags go through the videos API. For a broadcast,
    the video id is the same as the broadcast id.)
    """
    if yt_service is None or not yt_broadcast_id:
        return
    try:
        yt_service.videos().update(
            part="snippet",
            body={
                "id": yt_broadcast_id,
                "snippet": {
                    "title": YT_TITLE,
                    "description": YT_DESCRIPTION,
                    "tags": YT_TAGS,
                    "categoryId": YT_CATEGORY_ID,
                    "defaultLanguage": "ne",
                    "defaultAudioLanguage": "ne",
                },
            },
        ).execute()
        print("[youtube] ✅ Tags, category and language set.")
    except Exception as e:
        print(f"[youtube] ⚠️ Could not set tags/category (stream is unaffected): {e}")


def create_youtube_live():
    """
    Creates a YouTube broadcast + stream, binds them, and returns the RTMP URL
    (ingestion address + stream key). Auto-start / auto-stop are enabled, so the
    broadcast goes live as soon as FFmpeg starts sending data.
    """
    global yt_service, yt_broadcast_id

    print("\n[3/5] Contacting YouTube Live API...")

    yt_service = build_youtube_service()

    start_time_iso = datetime.datetime.now(datetime.timezone.utc).strftime(
        "%Y-%m-%dT%H:%M:%S.000Z"
    )

    broadcast = yt_service.liveBroadcasts().insert(
        part="snippet,status,contentDetails",
        body={
            "snippet": {
                "title": YT_TITLE,
                "description": YT_DESCRIPTION,
                "scheduledStartTime": start_time_iso,
            },
            "status": {
                "privacyStatus": YOUTUBE_PRIVACY,
                "selfDeclaredMadeForKids": False,
            },
            "contentDetails": {
                "enableAutoStart": True,
                "enableAutoStop": True,
                "enableDvr": True,
                "monitorStream": {"enableMonitorStream": False},
            },
        },
    ).execute()

    yt_broadcast_id = broadcast["id"]

    stream = yt_service.liveStreams().insert(
        part="snippet,cdn,contentDetails",
        body={
            "snippet": {"title": YT_TITLE},
            "cdn": {
                "frameRate": "variable",
                "ingestionType": "rtmp",
                "resolution": "variable",
            },
            "contentDetails": {"isReusable": False},
        },
    ).execute()

    yt_stream_id = stream["id"]

    yt_service.liveBroadcasts().bind(
        part="id,contentDetails",
        id=yt_broadcast_id,
        streamId=yt_stream_id,
    ).execute()

    # Tags, category, language
    update_youtube_metadata()

    ingestion = stream["cdn"]["ingestionInfo"]
    yt_stream_url = f"{ingestion['ingestionAddress']}/{ingestion['streamName']}"

    print("\n✅ YouTube Live Broadcast Created!")
    print(f"YouTube Broadcast ID: {yt_broadcast_id}")
    print(f"Watch URL: https://www.youtube.com/watch?v={yt_broadcast_id}")

    return yt_stream_url, yt_stream_id


def youtube_wait_until_live(yt_stream_id, timeout_seconds=90):
    """
    Waits for YouTube to receive data and go live. Auto-start normally handles
    this; if it doesn't, we try to transition the broadcast to 'live' ourselves.
    Never raises - problems here are printed as warnings.
    """
    if yt_service is None or not yt_broadcast_id:
        return

    deadline = time.time() + timeout_seconds
    tried_transition = False

    while time.time() < deadline:
        try:
            st = yt_service.liveStreams().list(
                part="status", id=yt_stream_id
            ).execute()
            stream_status = (st.get("items") or [{}])[0].get("status", {}).get("streamStatus")

            bc = yt_service.liveBroadcasts().list(
                part="status", id=yt_broadcast_id
            ).execute()
            life = (bc.get("items") or [{}])[0].get("status", {}).get("lifeCycleStatus")

            print(f"[youtube] stream={stream_status} broadcast={life}")

            if life == "live":
                print("[youtube] ✅ Broadcast is LIVE.")
                return

            if stream_status == "active" and life in ("ready", "testing") and not tried_transition:
                tried_transition = True
                try:
                    yt_service.liveBroadcasts().transition(
                        broadcastStatus="live",
                        id=yt_broadcast_id,
                        part="status",
                    ).execute()
                    print("[youtube] Requested transition to LIVE.")
                except Exception as e:
                    print(f"[youtube] Transition request failed (auto-start may still work): {e}")
        except Exception as e:
            print(f"[youtube] Status check error: {e}")

        time.sleep(5)

    print("[youtube] ⚠️ Did not confirm LIVE within the timeout. "
          "Check YouTube Studio; it may still go live shortly.")


def end_youtube_live():
    """Ends the YouTube broadcast (transition to 'complete')."""
    if yt_service is None or not yt_broadcast_id:
        return
    try:
        yt_service.liveBroadcasts().transition(
            broadcastStatus="complete",
            id=yt_broadcast_id,
            part="status",
        ).execute()
        print("YouTube Live ended.")
    except Exception as e:
        print(f"Could not explicitly end YouTube Live (it may auto-end): {e}")


# ==============================================================================
# 4. AUTO-DETECT GPU
# ==============================================================================

def gpu_available():
    has_nvidia_gpu = subprocess.run(
        "nvidia-smi", shell=True, capture_output=True
    ).returncode == 0

    if not has_nvidia_gpu:
        return False

    ffmpeg_has_nvenc = subprocess.run(
        "ffmpeg -hide_banner -encoders 2>/dev/null | "
        "grep -q h264_nvenc && echo YES || echo NO",
        shell=True,
        capture_output=True,
        text=True,
        executable="/bin/bash"
    ).stdout.strip() == "YES"

    return ffmpeg_has_nvenc


USE_GPU = gpu_available()
print(f"\nGPU detected: {USE_GPU}")
print(f"Mode: {MODE}")
if MODE in ("live", "both"):
    print(f"Stream to: {STREAM_TO}")

if USE_GPU:
    print("Using NVENC hardware encoding.")
else:
    print("No usable GPU found — falling back to libx264 (CPU) encoding.")


# ==============================================================================
# 5. START VIRTUAL DISPLAY + AUDIO
# ==============================================================================

print(f"\n[1/5] Starting Xvfb {DISPLAY_NUM} at {WIDTH}x{HEIGHT}...")

xvfb_proc = subprocess.Popen([
    "Xvfb", DISPLAY_NUM, "-screen", "0", f"{WIDTH}x{HEIGHT}x24"
])

os.environ["DISPLAY"] = DISPLAY_NUM
time.sleep(2)

print("Starting Matchbox Window Manager...")
wm_proc = subprocess.Popen(["matchbox-window-manager", "-use_titlebar", "no"])
time.sleep(1)

print("Starting PulseAudio...")
pulseaudio_proc = subprocess.Popen(["pulseaudio", "--start", "--exit-idle-time=-1"])
time.sleep(2)

subprocess.run("pactl unload-module module-null-sink 2>/dev/null || true", shell=True)

print("Creating VirtualSink...")
subprocess.run(
    "pactl load-module module-null-sink "
    "sink_name=VirtualSink "
    "sink_properties=device.description=VirtualSink",
    shell=True,
    check=True
)
subprocess.run("pactl set-default-sink VirtualSink", shell=True, check=True)


# ==============================================================================
# 6. MAIN
# ==============================================================================

async def main():

    global ffmpeg_proc, live_video_id

    async with async_playwright() as p:

        user_data_dir = tempfile.mkdtemp(prefix="quiz_live_")

        print("\nStarting Chromium...")

        context = await p.chromium.launch_persistent_context(
            user_data_dir=user_data_dir,
            headless=False,
            no_viewport=True,
            permissions=["microphone", "camera"],
            ignore_default_args=["--enable-automation"],
            args=[
                f"--window-size={WIDTH},{HEIGHT}",
                "--window-position=0,0",
                "--kiosk",
                "--start-fullscreen",
                "--no-first-run",
                "--no-default-browser-check",
                "--disable-infobars",
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--autoplay-policy=no-user-gesture-required",
                "--use-fake-ui-for-media-stream",
                "--use-fake-device-for-media-stream",
                "--force-device-scale-factor=1",
                "--disable-features=TranslateUI",
                "--overscroll-history-navigation=0",
                "--noerrdialogs",
                "--disable-session-crashed-bubble",
                "--disable-translate",
                "--disable-notifications",
            ],
        )

        page = context.pages[0] if context.pages else await context.new_page()

        # Keep the browser on a blank page while the slow setup (Facebook /
        # YouTube API calls) happens. The website is only opened afterwards,
        # and nothing is streamed until then.
        print("\n[2/5] Browser ready (blank page). Preparing live broadcasts first...")

        print("Making Chromium fullscreen...")
        subprocess.run('wmctrl -r ":ACTIVE:" -b add,fullscreen', shell=True)
        await asyncio.sleep(2)

        # ======================================================================
        # 3. CREATE LIVE BROADCASTS (Facebook and/or YouTube)
        # ======================================================================

        fb_stream_url = None
        yt_stream_url = None
        yt_stream_id = None

        if MODE in ("live", "both"):

            errors = []

            if USE_FACEBOOK:
                try:
                    fb_stream_url = create_facebook_live()
                except Exception as e:
                    errors.append(f"Facebook: {e}")
                    print(f"⚠️ Facebook setup failed: {e}")

            if USE_YOUTUBE:
                try:
                    yt_stream_url, yt_stream_id = create_youtube_live()
                except Exception as e:
                    errors.append(f"YouTube: {e}")
                    print(f"⚠️ YouTube setup failed: {e}")

            if not fb_stream_url and not yt_stream_url:
                raise RuntimeError(
                    "No live destination could be created. " + " | ".join(errors)
                )

            if errors:
                print("\n⚠️ Continuing with the destination(s) that worked.")

        else:
            print("\n[3/5] Record-only mode — skipping Facebook/YouTube.")

        # ======================================================================
        # 4. START FFMPEG
        # ======================================================================

        print("\n[4/5] Preparing FFmpeg command...")

        if USE_GPU:
            video_encode_args = [
                "-c:v", "h264_nvenc",
                "-preset", "p4",
                "-tune", "ll",
                "-b:v", "8000k",
                "-maxrate", "9000k",
                "-bufsize", "18000k",
                "-pix_fmt", "yuv420p",
                "-g", str(FRAMERATE * 2),
            ]
        else:
            video_encode_args = [
                "-c:v", "libx264",
                "-preset", "ultrafast",
                "-tune", "zerolatency",
                "-b:v", "4000k",
                "-maxrate", "4500k",
                "-bufsize", "9000k",
                "-pix_fmt", "yuv420p",
                "-g", str(FRAMERATE * 2),
            ]

        ffmpeg_cmd = [
            "ffmpeg", "-y",

            # Video input
            "-thread_queue_size", "512",
            "-f", "x11grab",
            "-video_size", f"{WIDTH}x{HEIGHT}",
            "-framerate", str(FRAMERATE),
            "-draw_mouse", "0",
            "-i", DISPLAY_NUM,

            # Audio input
            "-thread_queue_size", "512",
            "-f", "pulse",
            "-i", "VirtualSink.monitor",

            "-map", "0:v",
            "-map", "1:a",

            *video_encode_args,

            "-c:a", "aac",
            "-b:a", "128k",
            "-ar", "44100",

            "-flags", "+global_header",
        ]

        # Build the list of output destinations as (format, target)
        outputs = []
        if fb_stream_url:
            outputs.append(("flv", fb_stream_url))
        if yt_stream_url:
            outputs.append(("flv", yt_stream_url))
        if MODE in ("both", "record"):
            outputs.append(("matroska", RECORD_PATH))

        if len(outputs) == 1:
            # Single destination: exactly the same as the original behaviour
            fmt, target = outputs[0]
            ffmpeg_cmd += ["-f", fmt, target]
        else:
            # Multiple destinations: one encode, many outputs via tee.
            # onfail=ignore: if one platform drops, the others keep going.
            tee_parts = []
            for fmt, target in outputs:
                if fmt == "flv":
                    tee_parts.append(f"[f=flv:onfail=ignore]{target}")
                else:
                    tee_parts.append(f"[f=matroska]{target}")
            ffmpeg_cmd += ["-f", "tee", "|".join(tee_parts)]

        # ======================================================================
        # 4a. OPEN THE WEBSITE, THEN START STREAMING EXACTLY N SECONDS LATER
        # ======================================================================

        print("\n[2/5] Opening website...")
        print(f"Website URL: {WEBSITE_URL}")

        nav_start = time.time()
        await page.goto(WEBSITE_URL, wait_until="domcontentloaded", timeout=120000)

        try:
            await page.evaluate(
                "document.activeElement && document.activeElement.blur();"
            )
        except Exception:
            pass

        print(f"✅ Website opened ({time.time() - nav_start:.1f}s after navigation started).")

        wait_left = STREAM_START_DELAY_SECONDS - (time.time() - nav_start)
        if wait_left <= 0:
            print(
                f"⚠️ The page took longer than {STREAM_START_DELAY_SECONDS}s to open; "
                "starting the stream immediately."
            )
        while wait_left > 0:
            print(f"   🔵 Stream starts in {wait_left:.0f}s...")
            await asyncio.sleep(min(1, wait_left))
            wait_left = STREAM_START_DELAY_SECONDS - (time.time() - nav_start)

        print("\n[4/5] Starting FFmpeg now...")
        print("FFmpeg started (command hidden to protect the stream keys).")

        ffmpeg_proc = subprocess.Popen(
            ffmpeg_cmd,
            stdout=subprocess.DEVNULL,
            stderr=None
        )

        await asyncio.sleep(5)

        if ffmpeg_proc.poll() is not None:
            print("\n❌ FFmpeg EXITED IMMEDIATELY!")
            print("Exit code:", ffmpeg_proc.returncode)
            raise RuntimeError("FFmpeg failed to start. Check the FFmpeg error printed above.")

        print("✅ FFmpeg is running.")

        # YouTube: check in the background that it goes live
        yt_wait_task = None
        if yt_stream_url:
            print("\n[youtube] Checking in the background that YouTube goes live...")
            yt_wait_task = asyncio.get_event_loop().run_in_executor(
                None, youtube_wait_until_live, yt_stream_id
            )

        # ======================================================================
        # 4b. SET + CONFIRM THE "PLAYING" FLAG
        # ======================================================================

        print("\n[flag] Telling the website that recording/live has started...")
        flag_confirmed = set_and_confirm_flag_true()

        if not flag_confirmed:
            print(
                "[flag] ⚠️ Could not confirm the flag. Auto-stop may not work; "
                f"the {HARD_SAFETY_CAP_MINUTES}-minute safety cap will still apply."
            )

        if yt_wait_task is not None:
            await yt_wait_task

        # ======================================================================
        # 5. RUNNING
        # ======================================================================

        print("\n" + "=" * 70)
        print("🔴 BROADCAST / RECORDING IS RUNNING!")
        print("=" * 70)

        print(f"Mode: {MODE}")
        if MODE in ("live", "both"):
            dests = []
            if fb_stream_url:
                dests.append("Facebook")
            if yt_stream_url:
                dests.append("YouTube")
            print(f"Streaming to: {', '.join(dests)}")
        print(f"Title: {TOPIC}")
        print(f"Display: {WIDTH}x{HEIGHT}")
        print(f"Encoder: {'NVENC (GPU)' if USE_GPU else 'libx264 (CPU)'}")
        print(f"Website: {WEBSITE_URL}")
        if live_video_id:
            print(f"Facebook Live ID: {live_video_id}")
        if yt_broadcast_id:
            print(f"YouTube: https://www.youtube.com/watch?v={yt_broadcast_id}")
        if MODE in ("record", "both"):
            print(f"Recording to: {RECORD_PATH}")

        print(f"\nChecking '{FIRESTORE_FIELD}' every {POLL_INTERVAL_SECONDS}s.")
        print(f"Will stop when it becomes False (safety cap: {HARD_SAFETY_CAP_MINUTES} min).")
        print("You can also interrupt this cell manually at any time.")

        max_seconds = HARD_SAFETY_CAP_MINUTES * 60
        start_time = time.time()
        next_poll = start_time + POLL_INTERVAL_SECONDS

        while True:

            if ffmpeg_proc.poll() is not None:
                print("\n❌ FFmpeg stopped unexpectedly!")
                print("FFmpeg exit code:", ffmpeg_proc.returncode)
                break

            now = time.time()

            if now - start_time >= max_seconds:
                print(f"\n⏱️ Reached {HARD_SAFETY_CAP_MINUTES} minute safety cap — stopping.")
                break

            if now >= next_poll:
                next_poll = now + POLL_INTERVAL_SECONDS
                status = get_live_status()
                elapsed_min = (now - start_time) / 60
                print(f"[flag] t={elapsed_min:.1f} min → {FIRESTORE_FIELD} = {status}")

                # Only an explicit False ends the run. None (couldn't read)
                # is treated as "unknown, keep going".
                if status is False:
                    print("\n✅ Website finished (flag is False) — stopping now.")
                    await asyncio.sleep(END_GRACE_SECONDS)
                    break

            await asyncio.sleep(5)


# ==============================================================================
# 7. RUN
# ==============================================================================

try:
    await main()

except (KeyboardInterrupt, asyncio.CancelledError):
    print("\nStopping (manual interrupt)...")

except Exception as e:
    print(f"\n❌ Error: {e}")

finally:

    print("\nCleaning up...")

    # Tell the website/other programs that we're no longer running.
    # This covers normal finish, manual stop, and errors.
    if flag_was_set:
        if set_live_status(False):
            print("[flag] Set flag to False.")

    # Stop FFmpeg gracefully (SIGINT lets it finalize the recording file)
    if ffmpeg_proc is not None:
        try:
            if ffmpeg_proc.poll() is None:
                ffmpeg_proc.send_signal(signal.SIGINT)
            ffmpeg_proc.wait(timeout=60)
            print(f"FFmpeg stopped (exit code {ffmpeg_proc.returncode}).")
        except subprocess.TimeoutExpired:
            print("⚠️ FFmpeg did not finalize within 60 seconds; terminating it.")
            try:
                ffmpeg_proc.kill()
                ffmpeg_proc.wait(timeout=10)
            except Exception:
                print("⚠️ Could not confirm FFmpeg termination.")
        except Exception as e:
            print(f"⚠️ Could not stop FFmpeg cleanly: {e}")

    # Explicitly end the Facebook live so it doesn't linger
    if live_video_id and PAGE_ACCESS_TOKEN:
        try:
            requests.post(
                f"https://graph.facebook.com/v20.0/{live_video_id}",
                data={"end_live_video": "true", "access_token": PAGE_ACCESS_TOKEN},
                timeout=20,
            )
            print("Facebook Live ended.")
        except Exception as e:
            print(f"Could not explicitly end Facebook Live: {e}")

    # Explicitly end the YouTube live
    end_youtube_live()

    for proc in (wm_proc, xvfb_proc, pulseaudio_proc):
        if proc is not None:
            try:
                proc.terminate()
            except Exception:
                pass

    # Convert recording to MP4 (no re-encode, takes seconds), then upload
    if MODE in ("record", "both") and os.path.isfile(RECORD_PATH):
        raw_size = os.path.getsize(RECORD_PATH)
        if raw_size == 0:
            print(f"❌ Recording file is empty: {RECORD_PATH}")
        else:
            print(f"Recording finalized: {RECORD_PATH} ({raw_size / (1024 * 1024):.1f} MB)")
            mp4_path = os.path.splitext(RECORD_PATH)[0] + ".mp4"
            print("\nConverting recording to MP4...")
            conversion = subprocess.run(
                ["ffmpeg", "-y", "-i", RECORD_PATH, "-c", "copy",
                 "-movflags", "+faststart", mp4_path],
                check=False
            )

            if (conversion.returncode == 0 and os.path.isfile(mp4_path)
                    and os.path.getsize(mp4_path) > 0):
                size_mb = os.path.getsize(mp4_path) / (1024 * 1024)
                print(f"✅ Saved: {mp4_path} ({size_mb:.1f} MB)")

                if UPLOAD_ENABLED:
                    upload_video(mp4_path, DESCRIPTION, TITLE)
                else:
                    print("Upload disabled (UPLOAD_ENABLED = False).")

                print("Recording is also in the Output panel (right side) under /kaggle/working.")
            else:
                print(
                    f"❌ MP4 conversion failed (FFmpeg exit code {conversion.returncode}). "
                    f"The raw recording is at: {RECORD_PATH}"
                )
    elif MODE in ("record", "both"):
        print(
            f"❌ No recording file was created at {RECORD_PATH}. "
            "Check FFmpeg output errors and confirm /kaggle/working has free space."
        )

    print("Stopped cleanly.")

[setup    0s] apt: installing matchbox-window-manager, pulseaudio, pulseaudio-utils, wmctrl, dbus-x11, xdotool ...
[setup    0s] pip: installing playwright, nepali-datetime ...
[setup    8s] pip: done.
[setup    8s] playwright: downloading Chromium ...
[setup   22s] playwright: Chromium downloaded.
[setup   25s] apt: done.
[setup   25s] playwright: installing Chromium system libraries ...
[setup   52s] playwright: system libraries done.
[setup   52s] Setup finished.
[0/5] Cleaning previous display/stream processes...


pkill: pattern that searches for process name longer than 15 characters will result in zero matches
Try `pkill -f' option to match against the complete command line.


Title: गोरखापत्र कार्तिक ११ मा प्रकाशित सामान्य ज्ञान GK

GPU detected: True
Mode: live
Stream to: both
Using NVENC hardware encoding.

[1/5] Starting Xvfb :62 at 1080x1350...


The XKEYBOARD keymap compiler (xkbcomp) reports:
> Warning:          Could not resolve keysym XF86CameraAccessEnable
> Warning:          Could not resolve keysym XF86CameraAccessDisable
> Warning:          Could not resolve keysym XF86CameraAccessToggle
> Warning:          Could not resolve keysym XF86NextElement
> Warning:          Could not resolve keysym XF86PreviousElement
> Warning:          Could not resolve keysym XF86AutopilotEngageToggle
> Warning:          Could not resolve keysym XF86MarkWaypoint
> Warning:          Could not resolve keysym XF86Sos
> Warning:          Could not resolve keysym XF86NavChart
> Warning:          Could not resolve keysym XF86FishingChart
> Warning:          Could not resolve keysym XF86SingleRangeRadar
> Warning:          Could not resolve keysym XF86DualRangeRadar
> Warning:          Could not resolve keysym XF86RadarOverlay
> Warning:          Could not resolve keysym XF86TraditionalSonar
> Warning:          Could not resolve keysym XF86Clearvu

Starting Matchbox Window Manager...
Starting PulseAudio...


...


Cleaning up...


[flv @ 0x5d3f2874c440] Failed to update header with correct duration. drop=1 speed=   1x    
[flv @ 0x5d3f2874c440] Failed to update header with correct filesize.
[flv @ 0x5d3f28761a80] Failed to update header with correct duration.
[flv @ 0x5d3f28761a80] Failed to update header with correct filesize.
[out#0/tee @ 0x5d3f275bee40] video:607111kB audio:9631kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: unknown
frame=18601 fps= 30 q=8.0 Lsize=N/A time=00:10:20.00 bitrate=N/A dup=2 drop=1 speed=   1x    


[flag] Set flag to False.


[aac @ 0x5d3f275e04c0] Qavg: 656.725
Exiting normally, received signal 2.


Facebook Live ended.
YouTube Live ended.
Stopped cleanly.
